# 04_New_Models_Comparison.ipynb

This notebook implements and evaluates two new alternative models to address the J-shaped distribution of star ratings:
1. **Ordinal Logistic Regression** (Proportional Odds Model)
2. **Binary Logistic Regression** (5-stars vs. 1-4 stars)

We use the same 5-fold cross-validation setup (random_state=42) and GLASSO-selected interactions as the existing linear models to ensure comparability.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import pickle
import ast

# Add parent directory to path for imports
sys.path.append(os.path.abspath('..'))

from src.modeling_alternative import run_ordinal_cv, run_binary_cv, get_existing_results
from src.modeling import prepare_raw_modeling_data

# 1. Load Local Data
data_path = '../data/Seminar_Amazon_Results_FULL.csv'
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print(f"✅ Local data loaded: {len(df)} rows.")
    
    # Parse stringified lists of tuples if they are strings
    if isinstance(df['aspect_sentiments'].iloc[0], str):
        print("Parsing aspect_sentiments column...")
        df['aspect_sentiments'] = df['aspect_sentiments'].apply(ast.literal_eval)
else:
    print(f"❌ Error: File not found at {data_path}. Please ensure the CSV is in the './data/' folder.")

c:\Users\Nativ\Documents\seminar\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Local data loaded: 701528 rows.
Parsing aspect_sentiments column...


## Part 1: Ordinal Logistic Regression
We treat star ratings as ordered categories.

In [2]:
print("Running Ordinal Logistic Regression CV...")
ordinal_results = run_ordinal_cv(df)

# Save results
with open('../model_ordinal_results.pkl', 'wb') as f:
    pickle.dump(ordinal_results, f)

def summarize_ordinal(res_list):
    df_res = pd.DataFrame(res_list)
    return df_res.mean()

ord_add_summary = summarize_ordinal(ordinal_results['additive'])
ord_int_summary = summarize_ordinal(ordinal_results['interaction'])

print("\nOrdinal Additive Summary:")
print(ord_add_summary)
print("\nOrdinal Interaction Summary:")
print(ord_int_summary)

2026-05-27 16:57:08,018 - INFO - Pivoting raw sentiment features...


Running Ordinal Logistic Regression CV...


2026-05-27 16:57:09,034 - INFO - Sparsity Filter: Dropped 379395 empty reviews. Remaining valid reviews: 322133
2026-05-27 16:57:09,034 - INFO - Starting Ordinal Logistic Regression Cross-Validation...
2026-05-27 16:57:09,062 - INFO - Processing Fold 1/5...


KeyboardInterrupt: 

## Part 2: Binary Logistic Regression
We predict whether a review is a 5-star review (1) or not (0).

In [ ]:
print("Running Binary Logistic Regression CV...")
binary_results = run_binary_cv(df)

# Save results
with open('../model_binary_results.pkl', 'wb') as f:
    pickle.dump(binary_results, f)

def summarize_binary(res_list):
    df_res = pd.DataFrame(res_list)
    return df_res.mean()

bin_add_summary = summarize_binary(binary_results['additive'])
bin_int_summary = summarize_binary(binary_results['interaction'])

print("\nBinary Additive Summary:")
print(bin_add_summary)
print("\nBinary Interaction Summary:")
print(bin_int_summary)

## Part 3: Final Comparison Table
Consolidating all models.

In [ ]:
existing = get_existing_results()

summary_data = []

# Linear Models
summary_data.append({
    'Model': 'Linear Additive',
    'Type': 'Continuous',
    'Metric 1 (RMSE/Acc)': existing['Linear Additive']['Avg RMSE'],
    'Metric 2 (AdjR2/AUC)': existing['Linear Additive']['Avg Adj R2'],
    'Complexity (BIC/AIC)': existing['Linear Additive']['Full BIC']
})
summary_data.append({
    'Model': 'Linear Interaction',
    'Type': 'Continuous',
    'Metric 1 (RMSE/Acc)': existing['Linear Interaction']['Avg RMSE'],
    'Metric 2 (AdjR2/AUC)': existing['Linear Interaction']['Avg Adj R2'],
    'Complexity (BIC/AIC)': existing['Linear Interaction']['Full BIC']
})

# Ordinal Models
summary_data.append({
    'Model': 'Ordinal Additive',
    'Type': 'Categorical',
    'Metric 1 (RMSE/Acc)': ord_add_summary['rps'], # Using RPS for Metric 1
    'Metric 2 (AdjR2/AUC)': ord_add_summary['log_likelihood'],
    'Complexity (BIC/AIC)': ord_add_summary['aic']
})
summary_data.append({
    'Model': 'Ordinal Interaction',
    'Type': 'Categorical',
    'Metric 1 (RMSE/Acc)': ord_int_summary['rps'],
    'Metric 2 (AdjR2/AUC)': ord_int_summary['log_likelihood'],
    'Complexity (BIC/AIC)': ord_int_summary['aic']
})

# Binary Models
summary_data.append({
    'Model': 'Binary Additive',
    'Type': 'Binary',
    'Metric 1 (RMSE/Acc)': bin_add_summary['accuracy'],
    'Metric 2 (AdjR2/AUC)': bin_add_summary['roc_auc'],
    'Complexity (BIC/AIC)': bin_add_summary['f1']
})
summary_data.append({
    'Model': 'Binary Interaction',
    'Type': 'Binary',
    'Metric 1 (RMSE/Acc)': bin_int_summary['accuracy'],
    'Metric 2 (AdjR2/AUC)': bin_int_summary['roc_auc'],
    'Complexity (BIC/AIC)': bin_int_summary['f1']
})

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*90)
print("FINAL MODEL COMPARISON SUMMARY")
print("="*90)
print(summary_df.to_string(index=False))
print("="*90)

print("\nNote: Metrics vary by model type. Linear uses RMSE/AdjR2/BIC. Ordinal uses RPS/LL/AIC. Binary uses Acc/AUC/F1.")